# Compare E3SM Reanalysis / FOSIRL / 4DEnVarOcn Hindcasts with CESM-SMYLE

Extends the workflow and native seasonal-lead conventions in `2_refactor_leadtime_rmse_skill_map.ipynb`.

This version compares:

- `WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce`
- `WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL`
- `WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn`
- `CESM-SMYLE`

The E3SM and CESM-SMYLE comparisons are restricted to these initialization dates:

```python
1980050100, 1980110100, 1981050100, 1982050100, 1982110100, 1983050100, 1983110100
```

Because this is a very small sample, this notebook uses direct model-vs-observation RMSE. The maps and matched-ensemble bootstrap should be treated as diagnostic robustness comparisons, not definitive significance tests.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import xarray as xr 
import numpy as np  
import cftime
import copy
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import cartopy.crs as ccrs
import xesmf as xe
import xskillscore as xs
%matplotlib inline

# import ESP-Lab modules
#from esp_lab import data_access_smyle.py as data_access
from esp_lab import data_access_e3sm as data_access
from esp_lab import data_access_obs as obs_access
from esp_lab import stats

# import plotting and other utilities from esp_lab.utils (formerly SMYLEutils)
from esp_lab.utils import calendar_utils as cal
from esp_lab.utils import mapplot_utils as maps
from esp_lab.utils import colorbar_utils as cbars
from esp_lab.utils import regrid_utils as regrid
from esp_lab.utils import mov_utils as mov
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "jupyter" else Path.cwd()
WORKFLOWS_DIR = REPO_ROOT / "workflows"
if str(WORKFLOWS_DIR) not in sys.path:
    sys.path.insert(0, str(WORKFLOWS_DIR))

import rmse_compare_helper

# Default figure output directory for this HPC environment.
FIGURE_OUTDIR = Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag")
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)


def figure_filename(*parts, ext="png"):
    """Build consistent, readable lowercase snake_case figure filenames."""
    import re

    clean_parts = ["fig"]
    for part in parts:
        if part is None:
            continue
        text = str(part).strip()
        if not text:
            continue
        text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()
        if text:
            clean_parts.append(text)

    suffix = ext.lstrip(".").lower()
    return "_".join(clean_parts) + f".{suffix}"



In [ ]:
import dask
from dask.distributed import (
    wait,
    get_client
)
dask.__version__

## Preprocessing:  Data I/O using Dask

### Create Dask Cluster

In [ ]:
def get_ClusterClient(cluster_type='local', workers=4):
    """
    Create a Dask Cluster and Client based on the machine environment.
    cluster_type options: 'local', 'casper_pbs', 'slurm'
    """
    import dask
    from dask.distributed import Client
    
    # Optional global dask config
    dask.config.set({'array.slicing.split_large_chunks': True})

    if cluster_type == 'casper_pbs':
        # Specific to NCAR's Casper machine
        from dask_jobqueue import PBSCluster
        cluster = PBSCluster(
            cores=1,
            memory='20GB',
            processes=1,
            queue='casper',
            resource_spec='select=1:ncpus=1:mem=20GB',
            project='NCGD0011',
            walltime='02:00:00',
            interface='ib0'
        )
        dask.config.set({
            'distributed.dashboard.link':
            'https://jupyterhub.hpc.ucar.edu/stable/user/{USER}/proxy/{port}/status'
        })
        client = Client(cluster)
        cluster.scale(30)
        
    elif cluster_type == 'slurm':
        from dask_jobqueue import SLURMCluster
        cluster = SLURMCluster(
            cores=4,
            processes=1,
            memory="16GB",
            walltime="02:00:00"
            # queue="compute", 
            # project="my_project" 
        )
        client = Client(cluster)
        cluster.scale(workers)
        
    elif cluster_type == 'local':
        from dask.distributed import LocalCluster

        # Keep production worker requests from overwhelming the Jupyter host.
        local_worker_cap = int(os.environ.get("DASK_LOCAL_WORKERS", "2"))
        local_workers = max(1, min(workers, local_worker_cap))
        local_memory_limit = os.environ.get("DASK_LOCAL_MEMORY_LIMIT", "4GB")
        cluster = LocalCluster(
            n_workers=local_workers,
            threads_per_worker=1,
            processes=True,
            memory_limit=local_memory_limit,
            dashboard_address=None,
        )
        client = Client(cluster)
        print(
            f"Local Dask cluster: {local_workers} workers, "
            f"{local_memory_limit} limit per worker"
        )
        
    else:
        raise ValueError(f"Unknown cluster_type: {cluster_type}")

    return cluster, client

# Set this to 'casper_pbs' when on NCAR Casper, or 'local' on your PC
machine_env = os.environ.get('CLUSTER_TYPE', 'local') 

try:
    client = get_client()
    print(client)
except ValueError:
    print("No active client")
try:
    client.close()
    cluster.close()
except:
    pass

cluster, client = get_ClusterClient(cluster_type=machine_env, workers=30)

In [ ]:
cluster

### Read in EAM monthly data; Convert to Seasonal averages (DJF, MAM, JJA, SON)
- Chosen field is returned as a dask array with leading dimensions of Y (initialization year), M (ensemble member), and L (lead season). For example, for November starts, L=1 corresponds to first DJF season.
- "time" which gives prediction verification time (centered time for a given season) is also dimensioned with (Y,L)

In [ ]:
%%time
# -----------------------------------------------------------------------------
# Variable-driven setup + limited-date multi-case E3SM setup
# -----------------------------------------------------------------------------
# Change this one line to switch diagnostics:
#   field = "TREFHT"  -> ERA5 tas, model/obs K -> degC
#   field = "PRECT"   -> GPCP PRECT, model m/s -> mm/day
#   field = "PSL"     -> ERA5 psl, model/obs Pa -> hPa
#field = "PRECT"
#field = "TREFHT"
field = "PSL"

# Analysis-specific metadata lives in the notebook, not in rmse_compare_helper.
VAR_CONFIG = {
    "TREFHT": {
        "long_name": "2-m air temperature",
        "plot_name": "Surface air temperature",
        "obs_name": "ERA5",
        "obs_var": "tas",
        "obs_ys": "1979",
        "obs_ye": "2019",
        "model_convert": rmse_compare_helper.convert_kelvin_to_celsius,
        "smyle_convert": rmse_compare_helper.convert_kelvin_to_celsius,
        "obs_convert": rmse_compare_helper.convert_kelvin_to_celsius,
        "units": r"$^\circ$C",
    },
    "PRECT": {
        "long_name": "precipitation",
        "plot_name": "Precipitation",
        "obs_name": "GPCP_v2.3",
        "obs_var": "PRECT",
        "obs_ys": "1979",
        "obs_ye": "2017",
        "model_convert": rmse_compare_helper.convert_precip_mps_to_mmday,
        "smyle_convert": rmse_compare_helper.convert_precip_mps_to_mmday,
        "obs_convert": lambda da: rmse_compare_helper.no_unit_conversion(da, units="mm/day"),
        "units": "mm/day",
    },
    "PSL": {
        "long_name": "sea-level pressure",
        "plot_name": "Sea-level pressure",
        "obs_name": "ERA5",
        "obs_var": "psl",
        "obs_ys": "1979",
        "obs_ye": "2019",
        "model_convert": rmse_compare_helper.convert_pa_to_hpa,
        "smyle_convert": rmse_compare_helper.convert_pa_to_hpa,
        "obs_convert": rmse_compare_helper.convert_pa_to_hpa,
        "units": "hPa",
    },
}

if field not in VAR_CONFIG:
    raise ValueError(f"Unsupported field={field!r}. Available fields: {list(VAR_CONFIG)}")

cfg = VAR_CONFIG[field]

# Field-specific map colorbar configuration.
# `levels` control color boundaries; `ticks` control displayed labels.
MAP_COLORBAR_CONFIG = {
    "absolute_rmse": {
        "TREFHT": {"levels": [0, 0.5, 1, 1.5, 2, 2.5, 3],
                   "ticks": [0, 0.5, 1, 1.5, 2, 2.5, 3], "decimals": 1},
        "PRECT": {"levels": [0, 0.5, 1, 1.5, 2, 2.5, 3],
                  "ticks": [0, 0.5, 1, 1.5, 2, 2.5, 3], "decimals": 1},
        "PSL": {"levels": [0, 2, 4, 6, 8, 10, 12],
                "ticks": [0, 2, 4, 6, 8, 10, 12], "decimals": 1},
    },
    "rmse_difference": {
        "TREFHT": {
            "levels": [-4, -3.5, -3, -2.5, -2, -1.5, -1, -0.5,
                       0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4],
            "ticks": [-4, -3, -2, -1, 0, 1, 2, 3, 4], "decimals": 1,
        },
        "PRECT": {
            "levels": [-3, -2.5, -2, -1.5, -1, -0.5, 0,
                       0.5, 1, 1.5, 2, 2.5, 3],
            "ticks": [-3, -2, -1, 0, 1, 2, 3], "decimals": 1,
        },
        "PSL": {
            "levels": [-12, -10, -8, -6, -4, -2, 0,
                       2, 4, 6, 8, 10, 12],
            "ticks": [-12, -8, -4, 0, 4, 8, 12], "decimals": 1,
        },
    },
}


def map_colorbar_settings(plot_type, field):
    config = MAP_COLORBAR_CONFIG[plot_type][field]
    levels = np.asarray(config["levels"], dtype=float)
    ticks = np.asarray(config["ticks"], dtype=float)
    steps = np.diff(levels)
    if levels.ndim != 1 or levels.size < 2 or np.any(steps <= 0):
        raise ValueError(f"Invalid colorbar levels for {plot_type}/{field}")
    if not np.allclose(steps, steps[0]):
        raise ValueError(
            f"Map helper requires uniform levels for {plot_type}/{field}"
        )
    if np.any(ticks < levels[0]) or np.any(ticks > levels[-1]):
        raise ValueError(f"Ticks outside levels for {plot_type}/{field}")
    return levels, ticks, int(config["decimals"])


abs_rmse_levels, abs_rmse_ticks, abs_rmse_decimals = map_colorbar_settings(
    "absolute_rmse", field
)
diff_levels, diff_ticks, diff_decimals = map_colorbar_settings(
    "rmse_difference", field
)
print(f"Selected field: {field} ({cfg['long_name']})")
print(f"Observation product: {cfg['obs_name']} variable={cfg['obs_var']}")

# Prefer the newer project path, but keep the older location used by the PSL notebook as a fallback.
data_dir_candidates = [
    "/global/cfs/cdirs/e3sm/S2S2D/post_process",
    "/global/cfs/cdirs/e3smdata/simulations/S2S2D/post_process",
]
data_dir = next((p for p in data_dir_candidates if os.path.exists(p)), data_dir_candidates[0])
print(f"Using post-processed data directory: {data_dir}")

# -----------------------------------------------------------------------------
# NEW: multi-case E3SM setup
# -----------------------------------------------------------------------------
E3SM_CASES = {
    "E3SM-Reanalysis": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce",
        "cache_tag": "Reanalysis",
        "display_name": "Reanalysis",
    },
    "E3SM-FOSIRL": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
        "cache_tag": "JRA55_FOSIRL",
        "display_name": "FOSIRL",
    },
#    "E3SM-4DEnVarOcn": {
#        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn",
#        "cache_tag": "4DEnVarOcn",
#        "display_name": "4DEnVarOcn",
#    },
}

# ===== INITIALIZATION YEARS AND MONTHS CONFIGURATION =====
years = 1980
yeare = 2018
yexcl = None
init_months = [5, 11]  # May and November starts

lead_years = [y for y in np.arange(years, yeare + 1) if y != yexcl]
INIT_YEARS_BY_MONTH = {init_month: lead_years for init_month in init_months}

print(f"Hindcast period: {years}-{yeare}")
print(f"Initialization months: {init_months}")
print(f"Total hindcast years per month: {len(lead_years)}")

# Use the same years for de-drift and climatology
climy0 = years
climy1 = yeare

case_nens = 10
case_nlead = 24
engine = "netcdf4"
members = [f"EN{i:02d}" for i in range(case_nens)]

realm = "atm"
grid = "180x360_aave"
freq = "monthly"
ts_split = "2yr"

require_all_members = True
verify_field_name = True
verify_coverage = True
debug = True

# Optional: safer open-time chunking.
chunks_open = {} # {"L": 24} can be used if needed.

mchunk = {
    "Y": 3,
    "L": 24,
    "M": 2,
    "lat": 90,
    "lon": 180,
}

# ===== END CONFIGURATION BLOCK =====

e3sm_raw_by_case_month = {}

for case_key, case_info in E3SM_CASES.items():
    case_prefix = case_info["case_prefix"]
    e3sm_raw_by_case_month[case_key] = {}

    print("")
    print("=" * 80)
    print(f"Loading {case_key}: {case_prefix}")
    
    for init_month in init_months:
        lead_years = INIT_YEARS_BY_MONTH[init_month]
        init_tags = data_access.build_init_tags(lead_years, init_month)

        e3sm = data_access.get_monthly_data(
            data_dir=data_dir,
            case_prefix=case_prefix,
            members=members,
            init_tags=init_tags,
            field=field,
            nlead=case_nlead,
            chunks=chunks_open,
            realm=realm,
            grid=grid,
            freq=freq,
            ts_split=ts_split,
            require_all_members=require_all_members,
            verify_field_name=verify_field_name,
            verify_coverage=verify_coverage,
            engine=engine,
        )

        print(f"{case_key} init_month={init_month}, years={lead_years}, size={e3sm.nbytes / 1e9:.2f} GB")

        # Apply final chunking after combine.
        e3sm = e3sm.chunk(mchunk)

        if debug:
            print(e3sm)
            print(e3sm.dims)
            print(e3sm.Y.values)
            print(e3sm.L.values[:5], e3sm.L.values[-5:])
            print(e3sm.time.isel(Y=0, L=slice(0, 3)).values)

        e3sm_raw_by_case_month[case_key][init_month] = e3sm

# Backward-compatible aliases for older exploratory cells.
# The original notebook's "E3SMLE" is now mapped to the FOSIRL case.
e3smle_by_month = e3sm_raw_by_case_month["E3SM-FOSIRL"]
e3smle05 = e3smle_by_month.get(5)
e3smle11 = e3smle_by_month.get(11)

### Store datasets to disk for quicker processing next time (note this takes >30 minutes)

In [ ]:
%%time
# Store seasonal E3SM datasets to disk for quicker processing next time.
outdir = str(Path("/global/cfs/cdirs/e3sm/S2S2D/s2d_diag") / "JRA55_FOSIRL" / "leadtime_acc" / "work" / "limited_compare" / field)
os.makedirs(outdir, exist_ok=True)

date_tag = f"{INIT_DATES[0]}-{INIT_DATES[-1]}_n{len(INIT_DATES)}"

debug = False #True
force_rewrite = False
encoding = {
    field: {
        "chunksizes": (1, 8, 1, 90, 180),
        "zlib": True,
        "complevel": 1
    }
}

e3sm_seas_by_case_month = {}

for case_key, case_info in E3SM_CASES.items():
    cache_tag = case_info["cache_tag"]
    e3sm_seas_by_case_month[case_key] = {}

    for init_month in init_months:
        year_tag = "-".join(str(y) for y in INIT_YEARS_BY_MONTH[init_month])
        outname = (
            f"{cache_tag}_{init_month:02d}_{field}_N{case_nens:02d}_"
            f"M{case_nlead:02d}_years_{year_tag}_seas.nc"
        )
        outfile = os.path.join(outdir, outname)

        if os.path.exists(outfile) and not force_rewrite:
            e3sm_seas = xr.open_dataset(outfile, chunks=mchunk)
        else:
            if os.path.exists(outfile):
                os.remove(outfile)

            e3sm = e3sm_raw_by_case_month[case_key][init_month]

            # 1. seasonal aggregation (lazy)
            e3sm_seas = cal.mon_to_seas_dask(e3sm)

            # 2. rechunk after rolling
            e3sm_seas = e3sm_seas.chunk(mchunk)

            # 3. persist (optional but OK since reused)
            e3sm_seas = e3sm_seas.persist()

            # 4. write with encoding
            print(f"Writing {outfile}")
            e3sm_seas.to_netcdf(outfile, encoding=encoding)

            # 5. reopen from disk to keep graph small
            e3sm_seas = xr.open_dataset(outfile, chunks=mchunk)

        e3sm_seas_by_case_month[case_key][init_month] = e3sm_seas

        print(f"{case_key} init_month={init_month}: {outfile}")
        print(e3sm_seas)

# Backward-compatible alias.
e3smle_seas_by_month = e3sm_seas_by_case_month["E3SM-FOSIRL"]


### Regrid Hindcast data
- Regrid all hindcast outputs onto a common regular latitude–longitude grid (e.g., 5°×5°) to ensure consistent comparison across datasets
- Support multiple source grid types: a. Unstructured grids (e.g., E3SM/CAM-SE ncol) and b. tructured lat–lon grids (e.g., CESM/CAM-FV or post-processed E3SM)
- Apply a two-step workflow for unstructured grids: 1. Remap from native ncol to structured lat–lon using precomputed sparse mapping weights and 2. Regrid from intermediate lat–lon to target grid

In [ ]:
%%time
# Regrid all E3SM hindcast data to the common analysis grid.
target_dlat = 1.0
target_dlon = 1.0

destgrid = regrid.make_latlon_grid(dlat=target_dlat, dlon=target_dlon)
regrid_method = "conservative"
regrid_periodic = True

# Ensure clean chunking before regrid.
e3sm_seas_by_case_month = {
    case_key: {
        init_month: ds.chunk(mchunk)
        for init_month, ds in by_month.items()
    }
    for case_key, by_month in e3sm_seas_by_case_month.items()
}

# Build one E3SM regridder from the first case/month.
_ref_case = list(E3SM_CASES)[0]
_ref_month = init_months[0]
regridder = regrid.make_regridder(
    e3sm_seas_by_case_month[_ref_case][_ref_month],
    destgrid,
    method=regrid_method,
    periodic=regrid_periodic,
)

# Regrid, rechunk, and apply variable-specific unit conversion by case/init month.
e3sm_da_by_case_month = {}

for case_key in E3SM_CASES:
    e3sm_da_by_case_month[case_key] = {}

    for init_month in init_months:
        da = regridder(e3sm_seas_by_case_month[case_key][init_month][field]).chunk(mchunk)
        da = cfg["model_convert"](da)
        e3sm_da_by_case_month[case_key][init_month] = da

        print(f"{case_key} init_month={init_month}")
        print(da.shape)
        print(da.dims)
        print(da)

# Backward-compatible alias.
e3smle_da_by_month = e3sm_da_by_case_month["E3SM-FOSIRL"]

### Load CESM-SMYLE benchmark data
- Benchmark files are pre-generated once by `scripts/run_process_cesm_smyle_benchmark.py` and stored at `/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/`.
- Each file contains seasonal means on the native f09_g17 grid with dimensions `(Y, L, M, lat, lon)`.
- The benchmark is regridded here to the same analysis grid as E3SM before skill is computed.


In [ ]:
%%time
# Load and regrid CESM-SMYLE benchmark data to the same analysis grid as E3SM.
# The CESM-SMYLE requested dates/years are derived from the loaded E3SM data.

from esp_lab import data_access_cesm_smyle as smyle_access

SMYLE_BENCHMARK_DIR = "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE"
smyle_nens = 20
smyle_nlead = 24

mchunk_smyle = {"Y": 3, "L": -1, "M": 2, "lat": 96, "lon": 144}

# ---------------------------------------------------------------------
# Derive target CESM-SMYLE dates from the E3SM data already loaded.
# E3SM Y may be either:
#   1980        -> simple year
#   1980050100  -> full initialization date
# This block handles both.
# ---------------------------------------------------------------------
E3SM_REFERENCE_CASE = "E3SM-FOSIRL"

E3SM_YEARS_BY_MONTH = {}
E3SM_DATES_BY_MONTH = {}

for init_month in init_months:
    e3sm_y_raw = [
        int(yy)
        for yy in e3sm_raw_by_case_month[E3SM_REFERENCE_CASE][init_month]["Y"].values
    ]

    e3sm_dates = []
    e3sm_years = []

    for yy in e3sm_y_raw:
        yy_str = str(yy)

        if len(yy_str) == 4:
            # E3SM Y is simple year, e.g., 1980
            year = int(yy_str)
            ymdh = int(f"{year:04d}{init_month:02d}0100")

        elif len(yy_str) == 10:
            # E3SM Y is already full initialization date, e.g., 1980050100
            ymdh = int(yy_str)
            year = int(yy_str[:4])

        else:
            raise ValueError(
                f"Unexpected E3SM Y format: {yy}. "
                "Expected YYYY or YYYYMMDDHH."
            )

        e3sm_dates.append(ymdh)
        e3sm_years.append(year)

    E3SM_DATES_BY_MONTH[init_month] = e3sm_dates
    E3SM_YEARS_BY_MONTH[init_month] = e3sm_years

    print(f"\nE3SM reference case={E3SM_REFERENCE_CASE}, init_month={init_month}")
    print(f"  raw E3SM Y: {e3sm_y_raw}")
    print(f"  E3SM target dates for SMYLE: {e3sm_dates}")
    print(f"  E3SM target years for alignment: {e3sm_years}")

smyle_seas_by_month = {}
smyle_da_by_month = {}
SMYLE_MATCHED_DATES_BY_MONTH = {}
SMYLE_MATCHED_YEARS_BY_MONTH = {}

for init_month in init_months:
    ds = smyle_access.load_benchmark(
        field=field,
        init_month=init_month,
        benchmark_dir=SMYLE_BENCHMARK_DIR,
        nens=smyle_nens,
        nlead=smyle_nlead,
        freq="seas",
        chunks=mchunk_smyle,
    )

    # Keep original coordinate labels for .sel(), but use int labels for matching.
    available_y_original = list(ds["Y"].values)
    available_y_int = [int(yy) for yy in available_y_original]

    available_label_map = {
        int_label: original_label
        for int_label, original_label in zip(available_y_int, available_y_original)
    }

    requested_dates = [int(dd) for dd in E3SM_DATES_BY_MONTH[init_month]]

    matched_dates = [
        dd for dd in requested_dates
        if dd in available_label_map
    ]

    missing_dates = [
        dd for dd in requested_dates
        if dd not in available_label_map
    ]

    matched_y_labels = [
        available_label_map[dd]
        for dd in matched_dates
    ]

    matched_years = [
        int(str(dd)[:4])
        for dd in matched_dates
    ]

    print(f"\nCESM-SMYLE init_month={init_month}")
    print(f"  requested dates from E3SM: {requested_dates}")
    print(f"  available CESM-SMYLE dates: {available_y_int[0]}-{available_y_int[-1]}")
    print(f"  matched dates: {matched_dates}")
    print(f"  missing dates: {missing_dates}")
    print(f"  matched years: {matched_years}")
    print(f"  matched Y labels used for .sel(): {matched_y_labels}")

    if len(matched_dates) == 0:
        raise ValueError(
            f"No CESM-SMYLE dates matched for init_month={init_month}. "
            f"Requested from E3SM={requested_dates}, available={available_y_int}"
        )

    # Select with original coordinate labels to avoid dtype mismatch
    ds = ds.sel(Y=matched_y_labels).chunk(mchunk_smyle)

    smyle_seas_by_month[init_month] = ds
    SMYLE_MATCHED_DATES_BY_MONTH[init_month] = matched_dates
    SMYLE_MATCHED_YEARS_BY_MONTH[init_month] = matched_years

    print(f"  selected CESM-SMYLE sizes: {dict(ds.sizes)}")

# Build one CESM-SMYLE regridder using the first available initialization month
regridder_smyle = regrid.make_regridder(
    smyle_seas_by_month[init_months[0]],
    destgrid,
    method=regrid_method,
    periodic=regrid_periodic,
)

for init_month in init_months:
    da = regridder_smyle(smyle_seas_by_month[init_month][field]).chunk(mchunk)
    da = cfg["smyle_convert"](da)

    # Rename CESM-SMYLE Y from full date YYYYMMDDHH to simple year labels,
    # so later skill/anomaly code aligns with E3SM and observations.
    matched_years = SMYLE_MATCHED_YEARS_BY_MONTH[init_month]
    da = da.assign_coords(Y=("Y", matched_years))

    smyle_da_by_month[init_month] = da

    print(f"\nRegridded CESM-SMYLE init_month={init_month}")
    print(f"  original dates: {SMYLE_MATCHED_DATES_BY_MONTH[init_month]}")
    print(f"  renamed years:  {matched_years}")
    print(f"  shape: {da.shape}")
    print(f"  dims:  {da.dims}")
    print(da)

In [ ]:
for case_key, by_month in e3sm_da_by_case_month.items():
    for init_month, da in by_month.items():
        da.isel(Y=0, L=0, M=0).plot()
        plt.title(f"{case_key} init_month={init_month}")
        plt.show()

### Get Observational datasets
- Load observational datasets from the E3SM Diags archive (CMOR variables).
- Harmonize in time, variable naming, and space (regridding to a common grid).
- Provide consistent reference fields for hindcast evaluation.
- Compute OBS seasonal averages as a simple rolling mean
- Regrid to analysis grid 

In [ ]:
%%time
# Load observations using the selected field configuration.
obs_dir = "/global/cfs/cdirs/e3sm/e3sm_diags/obs_for_e3sm_diags/time-series"
obs_name = cfg["obs_name"]
obs_var = cfg["obs_var"]
obs_map = {field: obs_var}
obs_ys = cfg["obs_ys"]
obs_ye = cfg["obs_ye"]
verbose = True

obs_chunks = {
    "time": 24,
    "lat": 90,
    "lon": 180,
}

obs_monthly = obs_access.get_monthly_data(
    obs_dir=obs_dir,
    field=field,
    field_map=obs_map,
    product=obs_name,
    start_year=obs_ys,
    end_year=obs_ye,
    chunks=obs_chunks,
    verbose=verbose,
)

# Compute OBS seasonal averages.
obs_seas = obs_access.mon_to_seas_obs(
    obs_monthly,
    var=obs_var,
    field_map=obs_map,
)

# Rechunk after rolling.
obs_seas = obs_seas.chunk(obs_chunks)

# Build regridder from Dataset, not DataArray.
regridder_obs = regrid.make_regridder(
    obs_monthly,
    destgrid,
    method=regrid_method,
    periodic=regrid_periodic,
)

# Regrid seasonal DataArray.
try:
    obs_seas_rg = regridder_obs(
        obs_seas,
        output_chunks=obs_chunks,
    )
except TypeError:
    obs_seas_rg = regridder_obs(obs_seas)

# Rechunk after regrid and apply variable-specific unit conversion.
obs_seas_rg = obs_seas_rg.chunk(obs_chunks)
print(obs_seas_rg)

obs_seas_rg = cfg["obs_convert"](obs_seas_rg)
print(obs_seas_rg)

# Backward-compatible aliases for older exploratory cells.
if cfg["obs_name"] == "ERA5":
    obs_era5_seas_rg = obs_seas_rg
if field == "PRECT":
    obs_gpcp_seas_rg = obs_seas_rg

In [ ]:
obs_seas_rg.isel(time=100).plot()

# Direct RMSE Analysis for Limited Starts

With only 3–4 initialization years per month, correlation-based skill metrics (`corr`, `pval`, `MSSS`, `RPC`, etc.) are not robust and can become all-NaN.  
For this limited comparison, we directly compute model-vs-observation RMSE using the ensemble mean forecast:

`RMSE = sqrt(mean_Y((ensemble_mean(model) - obs)^2))`

We also save bias, MAE, and the number of valid years at each grid point.


In [ ]:
%%time
# Build model verification-time dictionaries.
# These are used to sample the observation seasonal means at the same verification seasons.

time_by_case_month = {
    case_key: {
        init_month: e3sm_seas_by_case_month[case_key][init_month].time.load()
        for init_month in init_months
    }
    for case_key in E3SM_CASES
}

# CESM-SMYLE time coordinate uses the selected full-date Y labels; convert them to simple years
# so it is consistent with smyle_da_by_month after we renamed Y to [1980, 1981, ...].
smyle_time_by_month = {}

for init_month in init_months:
    smyle_time = smyle_seas_by_month[init_month].time.load()
    smyle_time = smyle_time.assign_coords(
        Y=("Y", SMYLE_MATCHED_YEARS_BY_MONTH[init_month])
    )
    smyle_time_by_month[init_month] = smyle_time

# Backward-compatible convenience variables for exploratory cells.
time_by_month = time_by_case_month["E3SM-FOSIRL"]
e3smle05_time = time_by_month.get(5)
e3smle11_time = time_by_month.get(11)
smyle05_time = smyle_time_by_month.get(5)
smyle11_time = smyle_time_by_month.get(11)

print("Prepared verification-time arrays.")
for init_month in init_months:
    print(f"init_month={init_month}")
    print("  E3SM-FOSIRL time Y:", time_by_case_month["E3SM-FOSIRL"][init_month].Y.values)
    print("  CESM-SMYLE time Y:", smyle_time_by_month[init_month].Y.values)


In [ ]:
%%time
# Helper functions for direct RMSE.

DIRECT_RMSE_OUTDIR = str(Path("/global/cfs/cdirs/e3sm/S2S2D/s2d_diag") / "JRA55_FOSIRL" / "leadtime_acc" / "comparison" / "direct_rmse" / field)
os.makedirs(DIRECT_RMSE_OUTDIR, exist_ok=True)

# Native rolling-window coordinates from mon_to_seas_dask are 3, 6, ..., 24.
# L=3 verifies two months after initialization; figure Lead-1 means months 1-3.
DIRECT_RMSE_LEADS = list(rmse_compare_helper.DIRECT_RMSE_LEADS)

normalize_Y_to_year = rmse_compare_helper.normalize_y_to_year
drop_non_dim_coords = rmse_compare_helper.drop_non_dim_coords
get_ensemble_mean = rmse_compare_helper.get_ensemble_mean
make_obs_like_model = rmse_compare_helper.make_obs_like_model_time
prepare_model_for_direct_rmse = rmse_compare_helper.prepare_model_for_direct_rmse
finite_fraction = rmse_compare_helper.finite_fraction
normalize_direct_rmse_leads = rmse_compare_helper.normalize_direct_rmse_leads
require_available_lead = rmse_compare_helper.require_available_lead
safe_model_name = rmse_compare_helper.safe_model_name
area_weighted_mask_fraction = rmse_compare_helper.area_weighted_mask_fraction


def compute_direct_rmse(model_da, obs_da, model_time, common_years):
    return rmse_compare_helper.compute_direct_rmse(
        model_da=model_da,
        obs_da=obs_da,
        model_time=model_time,
        common_years=common_years,
        field=field,
        units=cfg["units"],
        lead_coord=DIRECT_RMSE_LEADS,
    )


def print_direct_rmse_input_check(label, model_da, obs_da, model_time, common_years):
    return rmse_compare_helper.print_direct_rmse_input_check(
        label=label,
        model_da=model_da,
        obs_da=obs_da,
        model_time=model_time,
        common_years=common_years,
    )

print("Direct-RMSE helper functions are ready.")


In [ ]:
%%time
# Compute and save direct RMSE for CESM-SMYLE and both E3SM cases.

force_compute_direct_rmse = False

direct_rmse_smyle_by_month = {}
direct_rmse_e3sm_by_case_month = {}

# Observation data used for direct RMSE.
obs_direct = obs_seas_rg.chunk({"time": 24, "lat": 90, "lon": 180})

# ---------------------------------------------------------------------
# CESM-SMYLE direct RMSE
# ---------------------------------------------------------------------
for init_month in init_months:
    common_years = SMYLE_MATCHED_YEARS_BY_MONTH[init_month]
    year_tag = "-".join(str(y) for y in common_years)

    out_nc = os.path.join(
        DIRECT_RMSE_OUTDIR,
        f"CESM-SMYLE_{field}_direct_rmse_init{init_month:02d}_years_{year_tag}.nc",
    )

    if os.path.exists(out_nc) and not force_compute_direct_rmse:
        ds_rmse = normalize_direct_rmse_leads(xr.open_dataset(out_nc).load())
    else:
        if force_compute_direct_rmse and os.path.exists(out_nc):
            os.remove(out_nc)

        print_direct_rmse_input_check(
            label=f"CESM-SMYLE direct RMSE input, init_month={init_month}",
            model_da=smyle_da_by_month[init_month],
            obs_da=obs_direct,
            model_time=smyle_time_by_month[init_month],
            common_years=common_years,
        )

        ds_rmse = compute_direct_rmse(
            model_da=smyle_da_by_month[init_month],
            obs_da=obs_direct,
            model_time=smyle_time_by_month[init_month],
            common_years=common_years,
        ).compute()

        ds_rmse.to_netcdf(out_nc)

    direct_rmse_smyle_by_month[init_month] = ds_rmse

    print("\nSaved/loaded:", out_nc)
    print(ds_rmse)


# ---------------------------------------------------------------------
# E3SM direct RMSE
# ---------------------------------------------------------------------
for case_key, case_info in E3SM_CASES.items():
    direct_rmse_e3sm_by_case_month[case_key] = {}

    for init_month in init_months:
        # Start from the E3SM-FOSIRL/SMYLE matched years, then intersect with this E3SM case.
        requested_years = SMYLE_MATCHED_YEARS_BY_MONTH[init_month]
        model_years = [
            int(str(y)[:4])
            for y in e3sm_da_by_case_month[case_key][init_month]["Y"].values
        ]
        common_years = [y for y in requested_years if y in model_years]

        if len(common_years) == 0:
            raise ValueError(f"No common years for {case_key}, init_month={init_month}")

        year_tag = "-".join(str(y) for y in common_years)
        safe_case = safe_model_name(case_key)

        out_nc = os.path.join(
            DIRECT_RMSE_OUTDIR,
            f"{safe_case}_{field}_direct_rmse_init{init_month:02d}_years_{year_tag}.nc",
        )

        if os.path.exists(out_nc) and not force_compute_direct_rmse:
            ds_rmse = normalize_direct_rmse_leads(xr.open_dataset(out_nc).load())
        else:
            if force_compute_direct_rmse and os.path.exists(out_nc):
                os.remove(out_nc)

            print_direct_rmse_input_check(
                label=f"{case_key} direct RMSE input, init_month={init_month}",
                model_da=e3sm_da_by_case_month[case_key][init_month],
                obs_da=obs_direct,
                model_time=time_by_case_month[case_key][init_month],
                common_years=common_years,
            )

            ds_rmse = compute_direct_rmse(
                model_da=e3sm_da_by_case_month[case_key][init_month],
                obs_da=obs_direct,
                model_time=time_by_case_month[case_key][init_month],
                common_years=common_years,
            ).compute()

            ds_rmse.to_netcdf(out_nc)

        direct_rmse_e3sm_by_case_month[case_key][init_month] = ds_rmse

        print("\nSaved/loaded:", out_nc)
        print(ds_rmse)


# Combined dictionary used by plotting cells.
direct_rmse_by_model = {
    "CESM-SMYLE": direct_rmse_smyle_by_month,
    **direct_rmse_e3sm_by_case_month,
}

# Backward-compatible aliases for exploratory cells.
skill_by_model = direct_rmse_by_model
skill_by_case_month = direct_rmse_e3sm_by_case_month
skill_by_month = direct_rmse_e3sm_by_case_month["E3SM-FOSIRL"]
smyle_skill_by_month = direct_rmse_smyle_by_month

print("Finished direct-RMSE calculation.")


In [ ]:
# Optional sanity check: direct-RMSE finite fractions

for model_name, by_month in direct_rmse_by_model.items():
    for init_month, ds in by_month.items():
        print("\n" + "=" * 80)
        print(f"{model_name}, init_month={init_month}")
        for var in ["rmse", "bias", "mae", "n_years"]:
            finite_fraction(ds[var], f"{var}")


In [ ]:
%%time
# Compute and save direct RMSE differences relative to CESM-SMYLE.
# Negative rmse_diff means E3SM has lower RMSE than CESM-SMYLE.

direct_rmse_diff_by_case_month = {}

for case_key in E3SM_CASES:
    direct_rmse_diff_by_case_month[case_key] = {}

    for init_month in init_months:
        ds_diff = rmse_compare_helper.direct_rmse_difference(
            direct_rmse_e3sm_by_case_month[case_key][init_month],
            direct_rmse_smyle_by_month[init_month],
        )
        ds_diff.attrs["description"] = (
            "Direct RMSE difference and ratio relative to CESM-SMYLE. "
            "Negative rmse_diff or rmse_ratio < 1 indicates smaller E3SM RMSE."
        )
        ds_diff.attrs["field"] = field
        ds_diff.attrs["units"] = cfg.get("units", "")

        direct_rmse_diff_by_case_month[case_key][init_month] = ds_diff

        year_tag = "-".join(str(y) for y in SMYLE_MATCHED_YEARS_BY_MONTH[init_month])
        safe_case = safe_model_name(case_key)

        out_nc = os.path.join(
            DIRECT_RMSE_OUTDIR,
            f"{safe_case}_minus_CESM-SMYLE_{field}_direct_rmse_diff_init{init_month:02d}_years_{year_tag}.nc",
        )

        ds_diff.to_netcdf(out_nc)

        print("\n" + "=" * 80)
        print(f"{case_key}, init_month={init_month}")
        print("Saved:", out_nc)
        print(ds_diff)

print("Finished direct-RMSE difference calculation.")


In [ ]:
# The original full skill-metric calculation is intentionally skipped for this limited-start notebook.
#
# With only 4 May starts and 3 November starts, ACC/p-values/MSSS/RPC are not robust.
# Use the direct-RMSE datasets produced above:
#
#   direct_rmse_by_model["CESM-SMYLE"][init_month]["rmse"]
#   direct_rmse_by_model["E3SM-FOSIRL"][init_month]["rmse"]
#   direct_rmse_by_model["E3SM-Reanalysis"][init_month]["rmse"]
#
# and direct-RMSE differences:
#
#   direct_rmse_diff_by_case_month[case_key][init_month]["rmse_diff"]


### Final Direct-RMSE Plot: CESM-SMYLE vs E3SM

- Reads or uses saved direct-RMSE files for CESM-SMYLE plus all configured E3SM cases.
- Plots each initialization month as grouped model columns.
- Uses direct model-vs-observation RMSE, not normalized RMSE from `compute_skill_seasonal`.


In [ ]:
%%time
# Plot direct absolute RMSE maps side by side:
# CESM-SMYLE plus all configured E3SM comparison sets.

from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
from matplotlib.offsetbox import AnchoredText

# ---------------------------------------------------------------------------
# User-adjustable final-plot setup
# ---------------------------------------------------------------------------
rmse_dir = DIRECT_RMSE_OUTDIR

metric = "Direct RMSE"
metric_units = cfg.get("units", "")
metric_label = f"{metric} ({metric_units})" if metric_units else metric

model_order = ["CESM-SMYLE", "E3SM-FOSIRL", "E3SM-Reanalysis", "E3SM-4DEnVarOcn"]
model_display_name = {
    "CESM-SMYLE": "CESM-SMYLE",
    "E3SM-FOSIRL": "E3SMv3-FOSIRL",
    "E3SM-Reanalysis": "E3SMv3-Reanalysis",
    "E3SM-4DEnVarOcn": "E3SMv3-4DEnVarOcn",
}

map_plot_leads = DIRECT_RMSE_LEADS[:-1]
season_names = ["DJF", "MAM", "JJA", "SON"]
init_month_names = {
    1: "JAN", 2: "FEB", 3: "MAR", 4: "APR", 5: "MAY", 6: "JUN",
    7: "JUL", 8: "AUG", 9: "SEP", 10: "OCT", 11: "NOV", 12: "DEC",
}

# Single font-size control for the full multi-panel figure.
fontz = 18
panel_title_size = fontz
lead_label_size = fontz * 0.85
lat_lon_label_size = fontz * 0.8
colorbar_label_size = fontz
colorbar_tick_size = fontz * 0.85
suptitle_size = fontz * 1.15
fweight = "bold"

# Plot style/layout controls.
figfmt = "png"
fig_col_width = 4.2
fig_row_height = 2.8
projection = ccrs.PlateCarree()
tight_layout_rect = [0.015, 0.13, 0.985, 0.94]
bottom = 0.10
hspace = 0.18
wspace = 0.08

cmap = "YlOrRd"
coff = 0.5

# Use the absolute-RMSE colorbar selected with `field` above.
colorbar_levels = abs_rmse_levels
colorbar_ticks = abs_rmse_ticks
colorbar_tick_decimals = abs_rmse_decimals

if colorbar_levels.ndim != 1 or colorbar_levels.size < 2:
    raise ValueError("colorbar_levels must contain at least two values")
level_steps = np.diff(colorbar_levels)
if np.any(level_steps <= 0) or not np.allclose(level_steps, level_steps[0]):
    raise ValueError(
        "colorbar_levels must be strictly increasing and uniformly spaced"
    )
if np.any(colorbar_ticks < colorbar_levels[0]) or np.any(
    colorbar_ticks > colorbar_levels[-1]
):
    raise ValueError("colorbar_ticks must lie within colorbar_levels")

cmin = float(colorbar_levels[0])
cmax_use = float(colorbar_levels[-1])
ci_use = float(level_steps[0])

colorbar_axes = [0.25, 0.055, 0.5, 0.02]
colorbar_orientation = "horizontal"
dpi = 300

lon_ticks = [-160, -80, 0, 80, 160]
lat_ticks = [-60, -30, 0, 30, 60]
lat_lon_tick_length = 2.5
lat_lon_tick_width = 0.5
map_axis_linewidth = 1.0
gridline_width = 0.35
gridline_color = "0.35"
gridline_alpha = 0.35
gridline_style = "-"

lead_label_loc = "lower right"
lead_label_pad = 0.25
lead_label_borderpad = 0.35
lead_label_bbox = dict(
    boxstyle="round,pad=0.2",
    facecolor="white",
    edgecolor="lightgray",
    alpha=0.85,
    linewidth=0.8,
)

# ---------------------------------------------------------------------------
# Load saved direct-RMSE files if the in-memory dictionary does not exist.
# ---------------------------------------------------------------------------
if "direct_rmse_by_model" not in globals():
    direct_rmse_by_model = {"CESM-SMYLE": {}, **{case_key: {} for case_key in E3SM_CASES}}

    for init_month in init_months:
        year_tag = "-".join(str(y) for y in SMYLE_MATCHED_YEARS_BY_MONTH[init_month])
        smyle_name = f"CESM-SMYLE_{field}_direct_rmse_init{init_month:02d}_years_{year_tag}.nc"
        direct_rmse_by_model["CESM-SMYLE"][init_month] = normalize_direct_rmse_leads(
            xr.open_dataset(os.path.join(rmse_dir, smyle_name)).load()
        )

        for case_key in E3SM_CASES:
            safe_case = safe_model_name(case_key)
            e3sm_name = f"{safe_case}_{field}_direct_rmse_init{init_month:02d}_years_{year_tag}.nc"
            direct_rmse_by_model[case_key][init_month] = normalize_direct_rmse_leads(
                xr.open_dataset(os.path.join(rmse_dir, e3sm_name)).load()
            )


def lead_label(init_month, lead):
    return rmse_compare_helper.lead_label(
        init_month, lead, init_month_names, season_names=season_names
    )


def add_lead_label(ax, text):
    label = AnchoredText(
        text,
        loc=lead_label_loc,
        prop=dict(size=lead_label_size, weight=fweight, family="monospace"),
        frameon=True,
        pad=lead_label_pad,
        borderpad=lead_label_borderpad,
    )
    label.patch.set(**lead_label_bbox)
    ax.add_artist(label)


def set_map_axis_linewidth(ax):
    for spine in ax.spines.values():
        spine.set_linewidth(map_axis_linewidth)
    if hasattr(ax, "outline_patch"):
        ax.outline_patch.set_linewidth(map_axis_linewidth)


def add_lat_lon_labels(ax, row, col, nrows, ncols):
    set_map_axis_linewidth(ax)
    ax.set_xticks(lon_ticks, crs=projection)
    ax.set_yticks(lat_ticks, crs=projection)
    ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True))
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(
        labelsize=lat_lon_label_size,
        length=lat_lon_tick_length,
        width=lat_lon_tick_width,
        top=False,
        right=False,
        labelbottom=(row == nrows - 1),
        labelleft=(col == 0),
    )
    ax.gridlines(
        crs=projection,
        linewidth=gridline_width,
        color=gridline_color,
        alpha=gridline_alpha,
        linestyle=gridline_style,
        draw_labels=False,
    )


rmse_plot_by_model = {
    model: {
        init_month: normalize_direct_rmse_leads(ds)
        for init_month, ds in by_month.items()
    }
    for model, by_month in direct_rmse_by_model.items()
}

plot_months = [
    init_month for init_month in init_months
    if all(init_month in rmse_plot_by_model[model] for model in model_order)
]

plot_leads = [
    lead for lead in map_plot_leads
    if all(
        lead in rmse_plot_by_model[model][init_month].L.values
        for model in model_order
        for init_month in plot_months
    )
]

nrows = len(plot_leads)
ncols = len(plot_months) * len(model_order)

subtitle = f"{cfg['plot_name']} direct RMSE, limited starts"
trendstr = "direct_rmse"

fig = plt.figure(figsize=(fig_col_width * ncols, fig_row_height * nrows))
fig.suptitle(subtitle, fontsize=suptitle_size, fontweight=fweight, y=0.995)

cntr = None
top_row_axes = {}

for i, lead in enumerate(plot_leads):
    for j, init_month in enumerate(plot_months):
        for k, model in enumerate(model_order):
            ds = rmse_plot_by_model[model][init_month]
            da = ds["rmse"].sel(L=lead)

            title = lead_label(init_month, lead)

            subplot = i * ncols + j * len(model_order) + k + 1

            ax, cntr = maps.map_pcolor_global_subplot(
                fig,
                da,
                da.lon,
                da.lat,
                ci_use,
                cmin,
                cmax_use,
                title,
                nrows,
                ncols,
                subplot,
                projection,
                cmap=cmap,
                cutoff=coff,
                fontsize=lead_label_size * 0.75,
            )

            col = j * len(model_order) + k
            if i == 0:
                top_row_axes[col] = ax
            add_lat_lon_labels(ax, i, col, nrows, ncols)

fig.tight_layout(rect=tight_layout_rect, h_pad=0.65, w_pad=0.35)
fig.subplots_adjust(bottom=bottom, hspace=hspace, wspace=wspace)
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
title_tops = [
    ax.title.get_window_extent(renderer=renderer)
    .transformed(fig.transFigure.inverted()).y1
    for ax in top_row_axes.values()
]
column_header_y = min(0.952, max(title_tops) + 0.012)
for j, init_month in enumerate(plot_months):
    for k, model in enumerate(model_order):
        col = j * len(model_order) + k
        ax_bbox = top_row_axes[col].get_position()
        model_name = model_display_name.get(model, model)
        fig.text(
            0.5 * (ax_bbox.x0 + ax_bbox.x1),
            column_header_y,
            model_name,
            ha="center",
            va="bottom",
            fontsize=panel_title_size,
            fontweight=fweight,
        )

cbar_ax = fig.add_axes(colorbar_axes)
cbar = fig.colorbar(cntr, cax=cbar_ax, orientation=colorbar_orientation)
cbar.set_ticks(colorbar_ticks)
cbar.ax.xaxis.set_major_formatter(
    mticker.FormatStrFormatter(f"%.{colorbar_tick_decimals}f")
)
cbar.set_label(metric_label, fontsize=colorbar_label_size, fontweight=fweight)
cbar.ax.tick_params(labelsize=colorbar_tick_size)

figname = figure_filename(field, "rmse_compare_global", ext=figfmt)
figpath = FIGURE_OUTDIR / figname
mov.save_figure(
    fig, figpath,
    mode="",
    metric="rmse_compare",
    title=f"RMSE Comparison: {field}",
    caption="RMSE comparison across experiments",
    dpi=dpi,
)
print("Saved figure:", figpath)
plt.show()

# Backward-compatible alias for the optional CONUS plot below.
skill_plot_by_model = rmse_plot_by_model

In [ ]:
%%time
# CONUS direct-RMSE comparison in one figure.
# Rows are verification seasons. Columns compare all models in forecast
# year 1, followed by all models in forecast year 2.

import cartopy.feature as cfeature
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
from matplotlib.colors import BoundaryNorm
import matplotlib.ticker as mticker

conus_models = ["CESM-SMYLE", "E3SM-FOSIRL", "E3SM-Reanalysis", "E3SM-4DEnVarOcn"]
conus_seasons = [
    ("JJA", 5, [3, 15]),
    ("SON", 5, [6, 18]),
    ("DJF", 11, [3, 15]),
    ("MAM", 11, [6, 18]),
]
conus_extent = [-125, -66, 24, 50]
conus_cmap = "YlOrRd"

# Use the absolute-RMSE colorbar selected with `field` above.
conus_colorbar_levels = abs_rmse_levels
conus_colorbar_ticks = abs_rmse_ticks
conus_colorbar_tick_labels = [
    f"{tick:.{abs_rmse_decimals}f}" for tick in conus_colorbar_ticks
]

# Figure size, font sizes, and spacing for the dense multi-column layout.
conus_col_width = 4.0
conus_row_height = 2.45

# Single typography control. Increase or decrease conus_fontz to scale all
# text while preserving the visual hierarchy among labels.
conus_fontz = 18
conus_suptitle_size = conus_fontz * 1.50
conus_column_header_size = conus_fontz * 0.96
conus_panel_title_size = conus_fontz * 0.83
conus_mean_label_size = conus_fontz * 0.625
conus_tick_size = conus_fontz * 0.67
conus_colorbar_label_size = conus_fontz * 1.17
conus_colorbar_tick_size = conus_fontz * 0.92

conus_hspace = 0.30
conus_wspace = 0.10
conus_tight_layout_rect = [0.015, 0.13, 0.995, 0.91]
conus_colorbar_axes = [0.27, 0.050, 0.46, 0.020]

if "rmse_plot_by_model" not in globals():
    if "skill_plot_by_model" in globals():
        rmse_plot_by_model = skill_plot_by_model
    else:
        raise RuntimeError(
            "Run the global direct-RMSE plotting cell first so "
            "rmse_plot_by_model is available."
        )


def subset_conus(da):
    if float(da.lon.max()) > 180:
        da = da.sel(lon=slice(235, 294))
        da = da.assign_coords(lon=(((da.lon + 180) % 360) - 180))
        da = da.sortby("lon")
    else:
        da = da.sel(lon=slice(conus_extent[0], conus_extent[1]))

    lat_slice = slice(conus_extent[2], conus_extent[3])
    if da.lat.size > 1 and float(da.lat[0]) > float(da.lat[-1]):
        lat_slice = slice(conus_extent[3], conus_extent[2])
    return da.sel(lat=lat_slice)


def add_conus_features(ax):
    for feature, kwargs in [
        (cfeature.COASTLINE, {"linewidth": 0.6}),
        (cfeature.BORDERS, {"linewidth": 0.4}),
        (cfeature.STATES, {"linewidth": 0.25, "edgecolor": "0.35"}),
    ]:
        try:
            ax.add_feature(feature, **kwargs)
        except Exception as err:
            print(f"Skipping Cartopy feature {feature}: {err}")


def conus_weighted_mean(da):
    weights = np.cos(np.deg2rad(da.lat))
    return float(da.weighted(weights).mean(("lat", "lon"), skipna=True))


# Collect every field first so availability and the shared color scale are
# validated before figure construction.
conus_fields = {model: {} for model in conus_models}
for model in conus_models:
    for season_name, init_month, season_leads in conus_seasons:
        if init_month not in rmse_plot_by_model.get(model, {}):
            raise KeyError(f"Missing {model} direct RMSE for init_month={init_month}.")

        rmse = rmse_plot_by_model[model][init_month]["rmse"]
        conus_fields[model][season_name] = {}
        for lead in season_leads:
            lead = require_available_lead(rmse, lead)
            conus_fields[model][season_name][lead] = subset_conus(rmse.sel(L=lead))

if conus_colorbar_levels.ndim != 1 or conus_colorbar_levels.size < 2:
    raise ValueError("conus_colorbar_levels must contain at least two values.")
if np.any(np.diff(conus_colorbar_levels) <= 0):
    raise ValueError("conus_colorbar_levels must be strictly increasing.")
if len(conus_colorbar_ticks) != len(conus_colorbar_tick_labels):
    raise ValueError(
        "conus_colorbar_ticks and conus_colorbar_tick_labels must match."
    )
if np.any(conus_colorbar_ticks < conus_colorbar_levels[0]) or np.any(
    conus_colorbar_ticks > conus_colorbar_levels[-1]
):
    raise ValueError("conus_colorbar_ticks must lie within the colorbar levels.")

conus_cmap_obj = plt.get_cmap(conus_cmap)
conus_norm = BoundaryNorm(conus_colorbar_levels, conus_cmap_obj.N, clip=True)
conus_plot_columns = [
    (model, year_index)
    for year_index in range(2)
    for model in conus_models
]

nrows = len(conus_seasons)
ncols = len(conus_plot_columns)
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(conus_col_width * ncols, conus_row_height * nrows),
    subplot_kw={"projection": ccrs.PlateCarree()},
    squeeze=False,
)

cntr = None
top_row_axes = {}
for i, (season_name, init_month, season_leads) in enumerate(conus_seasons):
    for j, (model, year_index) in enumerate(conus_plot_columns):
        lead = season_leads[year_index]
        da = conus_fields[model][season_name][lead]
        ax = axes[i, j]
        cntr = ax.pcolormesh(
            da.lon,
            da.lat,
            da,
            shading="nearest",
            cmap=conus_cmap_obj,
            norm=conus_norm,
            rasterized=True,
            transform=ccrs.PlateCarree(),
        )
        ax.set_extent(conus_extent, crs=ccrs.PlateCarree())
        add_conus_features(ax)
        ax.set_title(
            lead_label(init_month, lead),
            fontsize=conus_panel_title_size,
            fontweight=fweight,
        )
        if i == 0:
            top_row_axes[j] = ax
        ax.text(
            0.98,
            0.04,
            f"CONUS mean: {conus_weighted_mean(da):.2f}",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=conus_mean_label_size,
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="lightgray", alpha=0.85, linewidth=0.8),
        )
        ax.set_xticks([-120, -105, -90, -75], crs=ccrs.PlateCarree())
        ax.set_yticks([25, 35, 45], crs=ccrs.PlateCarree())
        ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True))
        ax.yaxis.set_major_formatter(LatitudeFormatter())
        ax.tick_params(
            labelsize=conus_tick_size,
            labelbottom=(i == nrows - 1),
            labelleft=(j == 0),
            top=False,
            right=False,
        )

fig.suptitle(
    f"{cfg['plot_name']} CONUS direct RMSE: limited-start comparison",
    fontsize=conus_suptitle_size,
    fontweight=fweight,
    y=0.985,
)
fig.tight_layout(rect=conus_tight_layout_rect, h_pad=0.8, w_pad=0.6)
fig.subplots_adjust(hspace=conus_hspace, wspace=conus_wspace)

# Place model/year headers above the panel titles, following notebook 2.
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
title_tops = [
    ax.title.get_window_extent(renderer=renderer)
    .transformed(fig.transFigure.inverted()).y1
    for ax in top_row_axes.values()
]
column_header_y = min(0.935, max(title_tops) + 0.010)
for j, (model, year_index) in enumerate(conus_plot_columns):
    ax_bbox = top_row_axes[j].get_position()
    fig.text(
        0.5 * (ax_bbox.x0 + ax_bbox.x1),
        column_header_y,
        f"{model_display_name.get(model, model)} | year {year_index + 1}",
        ha="center",
        va="bottom",
        fontsize=conus_column_header_size,
        fontweight=fweight,
    )

cbar_ax = fig.add_axes(conus_colorbar_axes)
cbar = fig.colorbar(cntr, cax=cbar_ax, orientation="horizontal")
cbar.set_ticks(conus_colorbar_ticks)
cbar.set_ticklabels(conus_colorbar_tick_labels)
cbar.set_label(
    metric_label,
    fontsize=conus_colorbar_label_size,
    fontweight=fweight,
)
cbar.ax.tick_params(labelsize=conus_colorbar_tick_size)

figname = figure_filename(field, "rmse_compare_conus", ext=figfmt)
figpath = FIGURE_OUTDIR / figname
mov.save_figure(
    fig, figpath,
    mode="",
    metric="rmse_compare",
    title=f"RMSE Comparison: {field}",
    caption="RMSE comparison across experiments",
    dpi=dpi,
)
print("Saved figure:", figpath)
plt.show()


### Finite-ensemble RMSE comparison following `1_ref*`

The original full skill-metric significance test is not appropriate for this limited-start notebook because the sample has only 4 May starts and 3 November starts.

For E3SM-FOSIRL, E3SM-Reanalysis, and E3SM-4DEnVarOcn versus CESM-SMYLE, this follows the finite-ensemble method in `1_ref*`: hold the smaller 10-member E3SM ensemble fixed and repeatedly sample the 20-member CESM-SMYLE ensemble down to 10 members without replacement. Verification years are held fixed.

This method cannot provide a finite-ensemble significance test for E3SM-Reanalysis versus E3SM-FOSIRL because both ensembles have 10 members. That comparison is plotted without significance markers. With only 3–4 verification years, the CESM-SMYLE comparison remains sensitivity evidence rather than definitive statistical significance.


### E3SM minus CESM-SMYLE direct-RMSE difference

This plots `E3SM direct RMSE - CESM-SMYLE direct RMSE` for all configured E3SM cases using the limited requested starts.

Interpretation:

- Negative values: E3SM has lower RMSE than CESM-SMYLE.
- Positive values: E3SM has higher RMSE than CESM-SMYLE.


In [ ]:
%%time
# Finite-ensemble RMSE comparison following the method in 1_ref*.
# Saves compact probability files used by the final comparison figures.
#
# Negative rmse_diff means the left-hand model has lower RMSE.
#
# The smaller E3SM ensemble is fixed. The larger CESM-SMYLE ensemble is
# repeatedly sampled without replacement to the E3SM member count. Years are
# held fixed, matching the finite-ensemble comparison used for ACC in 1_ref*.

import os
import glob
import numpy as np
import xarray as xr

BOOT_RMSE_OUTDIR = os.path.join(DIRECT_RMSE_OUTDIR, "bootstrap_rmse_diff")
os.makedirs(BOOT_RMSE_OUTDIR, exist_ok=True)

# Start with 100 for debugging. Increase to 500 or 1000 for final figures.
N_RESAMPLES = 1000
NBOOT = N_RESAMPLES  # compatibility alias used by downstream plotting cells
BOOT_RANDOM_SEED = 42
BOOT_ALPHA = 0.1
FORCE_COMPUTE_BOOTSTRAP = False

# Optional: restrict which leads to bootstrap to reduce cost.
# Use plot_leads/map_plot_leads if available; otherwise use all SMYLE leads.
BOOT_LEADS = None


# ---------------------------------------------------------------------
# Prepare observation data
# ---------------------------------------------------------------------
# obs_seas_rg is time-dimensional: (time, lat, lon)
# Convert it to hindcast-style dimensions: (Y, L, lat, lon)

obs_compare_chunks = {
    k: v for k, v in {
        "time": 24,
        "lat": 90,
        "lon": 180,
    }.items()
    if k in obs_seas_rg.dims
}

obs_compare = obs_seas_rg.chunk(obs_compare_chunks)

print("Created raw obs_compare:")
print(obs_compare)
print("obs_compare dims:", obs_compare.dims)


# Shared helper builds Y,L observations using the native seasonal convention:
# L=3 verifies at initialization month + 2 months, L=6 at +5 months, etc.
obs_compare_by_month = {}
for init_month in init_months:
    years = SMYLE_MATCHED_YEARS_BY_MONTH[init_month]
    template = smyle_da_by_month[init_month]
    leads = template["L"].values if BOOT_LEADS is None else BOOT_LEADS

    obs_compare_by_month[init_month] = rmse_compare_helper.make_obs_like_model_leads(
        obs_da=obs_compare,
        template_da=template,
        init_month=init_month,
        years=years,
        leads=leads,
    )

    print(f"\nOBS converted for init_month={init_month}")
    print(obs_compare_by_month[init_month])
    print("Y:", obs_compare_by_month[init_month]["Y"].values)
    print("L:", obs_compare_by_month[init_month]["L"].values)


# ---------------------------------------------------------------------
# Shared finite-ensemble helpers
# ---------------------------------------------------------------------
prepare_member_error = rmse_compare_helper.prepare_member_error
finite_ensemble_rmse_comparison_memorysafe = (
    rmse_compare_helper.finite_ensemble_rmse_comparison_memorysafe
)


# ---------------------------------------------------------------------
# The 1_ref* method applies to all 10-member E3SM ensembles when each is
# compared with the 20-member CESM-SMYLE ensemble.
# ---------------------------------------------------------------------
BOOT_COMPARISONS = [
    ("E3SM-FOSIRL", "CESM-SMYLE"),
    ("E3SM-Reanalysis", "CESM-SMYLE"),
    ("E3SM-4DEnVarOcn", "CESM-SMYLE"),
]


def bootstrap_model_data(model, init_month):
    if model == "CESM-SMYLE":
        return smyle_da_by_month[init_month]
    return e3sm_da_by_case_month[model][init_month]


bootstrap_rmse_diff_by_pair_month = {}

for left_model, right_model in BOOT_COMPARISONS:
    pair_key = f"{left_model}_minus_{right_model}"
    bootstrap_rmse_diff_by_pair_month[pair_key] = {}

    for init_month in init_months:
        common_years = SMYLE_MATCHED_YEARS_BY_MONTH[init_month]
        obs_da_this_month = obs_compare_by_month[init_month]

        safe_left = safe_model_name(left_model)
        safe_right = safe_model_name(right_model)
        year_tag = "-".join(str(y) for y in common_years)
        out_nc = os.path.join(
            BOOT_RMSE_OUTDIR,
            f"{safe_left}_minus_{safe_right}_{field}_finite_ensemble_resampling_"
            f"init{init_month:02d}_years_{year_tag}_niter{N_RESAMPLES}.nc",
        )

        if os.path.exists(out_nc) and not FORCE_COMPUTE_BOOTSTRAP:
            print("\n" + "=" * 80)
            print("Using existing bootstrap file:", out_nc)
            bootstrap_rmse_diff_by_pair_month[pair_key][init_month] = normalize_direct_rmse_leads(
                xr.open_dataset(out_nc).load()
            )
            continue

        left_err = prepare_member_error(
            model_da=bootstrap_model_data(left_model, init_month),
            obs_da=obs_da_this_month,
            common_years=common_years,
        )

        right_err = prepare_member_error(
            model_da=bootstrap_model_data(right_model, init_month),
            obs_da=obs_da_this_month,
            common_years=common_years,
        )

        print("\n" + "=" * 80)
        print(f"{left_model} minus {right_model}, init_month={init_month}")
        print("common_years:", common_years)
        print("obs dims:", obs_da_this_month.dims, obs_da_this_month.shape)
        print("obs Y:", obs_da_this_month["Y"].values)
        print("obs L:", obs_da_this_month["L"].values)
        print("left error dims:", left_err.dims, left_err.shape)
        print("right error dims:", right_err.dims, right_err.shape)

        ds_boot = finite_ensemble_rmse_comparison_memorysafe(
            left_err=left_err,
            right_err=right_err,
            n_iterations=N_RESAMPLES,
            seed=BOOT_RANDOM_SEED,
            alpha=BOOT_ALPHA,
        )

        ds_boot.attrs["left_model"] = left_model
        ds_boot.attrs["right_model"] = right_model
        bootstrap_rmse_diff_by_pair_month[pair_key][init_month] = ds_boot
        ds_boot.to_netcdf(out_nc)

        print("Saved:", out_nc)
        print(ds_boot)


In [ ]:
%%time
# Plot each configured E3SM case direct RMSE minus CESM-SMYLE direct RMSE.
# One figure per initialization month.
#
# Negative values mean E3SM has smaller model-vs-observation RMSE.
#
# Optional stippling:
#   STIPPLE_MODE = "probability":
#       stippling where prob_left_lower_rmse >= PROB_STIPPLE_THRESHOLD
#   STIPPLE_MODE = "significant":
#       stippling where left_better == 1

from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
import glob
import os
import numpy as np
import xarray as xr

diff_cases = ["E3SM-FOSIRL", "E3SM-Reanalysis", "E3SM-4DEnVarOcn"]
latlim = 80

# ---------------------------------------------------------------------
# Customizable colorbar range for RMSE difference
# ---------------------------------------------------------------------
# For RMSE difference:
#   negative = E3SM lower RMSE than CESM-SMYLE
#   positive = E3SM higher RMSE than CESM-SMYLE
# Use the RMSE-difference colorbar selected with `field` above.
cmin = float(diff_levels[0])
cmax = float(diff_levels[-1])
ci = float(np.diff(diff_levels)[0])

# ---------------------------------------------------------------------
# Optional finite-ensemble resampling markers
# ---------------------------------------------------------------------
PLOT_BOOTSTRAP_STIPPLING = True

# Probability is the fraction of matched-size ensemble samples in which the
# left-hand model has lower RMSE than the right-hand model.
STIPPLE_MODE = "significant"   # "probability" or "significant"
PROB_STIPPLE_THRESHOLD = 0.8    # try 0.8 first; use 0.9 for stricter stippling

BOOT_RMSE_OUTDIR = os.path.join(DIRECT_RMSE_OUTDIR, "bootstrap_rmse_diff")

# If NBOOT exists in the notebook, use it.
# If not, this block searches for any matching _nboot*.nc file.
BOOT_NBOOT_FOR_PLOT = globals().get("NBOOT", None)

# Stippling density / visibility
STIPPLE_STRIDE_LAT = 3
STIPPLE_STRIDE_LON = 3
STIPPLE_MARKER_SIZE = 2.0
STIPPLE_ALPHA = 0.8

# Single typography control for this dense difference-map figure.
compare_fontz = 18
compare_suptitle_size = compare_fontz * 1.1
compare_column_header_size = compare_fontz * 1.00
compare_panel_title_size = compare_fontz * 0.95
compare_lat_lon_label_size = compare_fontz * 0.95
compare_colorbar_label_size = compare_fontz * 0.95
compare_colorbar_tick_size = compare_fontz * 0.90
compare_fweight = "bold"

# Figure geometry and spacing.
compare_col_width = 5.35
compare_row_height = 2.55
compare_hspace = 0.25
compare_wspace = 0.060
# [left, bottom, right, top] reserved for maps and panel titles.
# These controls affect only the outer title/colorbar gaps.
compare_tight_layout_rect = [0.015, 0.070, 0.995, 0.965]
compare_suptitle_y = 0.988
compare_column_header_pad = 0.006
compare_column_header_y_max = 0.975
compare_colorbar_axes = [0.30, 0.032, 0.40, 0.018]

# Publication-style sparse map tick labels.
compare_lon_ticks = [-160, -80, 0, 80, 160]
compare_lon_tick_labels = [
    r"160$^\circ$W",
    r"80$^\circ$W",
    r"0$^\circ$",
    r"80$^\circ$E",
    r"160$^\circ$E",
]
compare_lat_ticks = [-60, -30, 0, 30, 60]
compare_lat_tick_labels = [
    r"60$^\circ$S",
    r"30$^\circ$S",
    r"0$^\circ$",
    r"30$^\circ$N",
    r"60$^\circ$N",
]

# ---------------------------------------------------------------------
# Load direct RMSE difference files if needed
# ---------------------------------------------------------------------
if "direct_rmse_diff_by_case_month" not in globals():
    direct_rmse_diff_by_case_month = {case_key: {} for case_key in diff_cases}

    for case_key in diff_cases:
        safe_case = safe_model_name(case_key)

        for init_month in init_months:
            year_tag = "-".join(str(y) for y in SMYLE_MATCHED_YEARS_BY_MONTH[init_month])

            fname = (
                f"{safe_case}_minus_CESM-SMYLE_{field}_direct_rmse_diff_"
                f"init{init_month:02d}_years_{year_tag}.nc"
            )

            fpath = os.path.join(DIRECT_RMSE_OUTDIR, fname)

            if not os.path.exists(fpath):
                print("Direct RMSE-difference file not found:", fpath)
                continue

            direct_rmse_diff_by_case_month[case_key][init_month] = normalize_direct_rmse_leads(
                xr.open_dataset(fpath).load()
            )

            print("Loaded direct RMSE-difference file:", fpath)

# ---------------------------------------------------------------------
# Load finite-ensemble probability files if requested and available
# ---------------------------------------------------------------------
bootstrap_rmse_diff_by_case_month = {}

if PLOT_BOOTSTRAP_STIPPLING:
    for case_key in diff_cases:
        bootstrap_rmse_diff_by_case_month[case_key] = {}
        safe_case = safe_model_name(case_key)

        for init_month in init_months:
            year_tag = "-".join(str(y) for y in SMYLE_MATCHED_YEARS_BY_MONTH[init_month])

            if BOOT_NBOOT_FOR_PLOT is None:
                pattern = os.path.join(
                    BOOT_RMSE_OUTDIR,
                    f"{safe_case}_minus_CESM-SMYLE_{field}_finite_ensemble_resampling_"
                    f"init{init_month:02d}_years_{year_tag}_niter*.nc",
                )
                matches = sorted(glob.glob(pattern))
                fpath = matches[-1] if matches else None
            else:
                fname = (
                    f"{safe_case}_minus_CESM-SMYLE_{field}_finite_ensemble_resampling_"
                    f"init{init_month:02d}_years_{year_tag}_niter{BOOT_NBOOT_FOR_PLOT}.nc"
                )
                fpath = os.path.join(BOOT_RMSE_OUTDIR, fname)

            if fpath is not None and os.path.exists(fpath):
                bootstrap_rmse_diff_by_case_month[case_key][init_month] = normalize_direct_rmse_leads(
                    xr.open_dataset(fpath).load()
                )
                print("Loaded bootstrap file:", fpath)
            else:
                print(
                    "Bootstrap file not found, skip stippling for:",
                    f"{safe_case}, init_month={init_month}, years={year_tag}",
                )

# ---------------------------------------------------------------------
# Determine months/leads to plot
# ---------------------------------------------------------------------
plot_months = [
    init_month for init_month in init_months
    if all(
        init_month in direct_rmse_diff_by_case_month.get(case_key, {})
        for case_key in diff_cases
    )
]

plot_leads = [
    lead for lead in map_plot_leads
    if all(
        lead in direct_rmse_diff_by_case_month[case_key][init_month].L.values
        for case_key in diff_cases
        for init_month in plot_months
    )
]

if len(plot_months) == 0:
    raise ValueError("No initialization months available for plotting.")

if len(plot_leads) == 0:
    raise ValueError("No leads available for plotting.")

print("plot_months:", plot_months)
print("plot_leads:", plot_leads)

proj = ccrs.PlateCarree()

# ---------------------------------------------------------------------
# Colorbar range
# ---------------------------------------------------------------------
print(f"Colorbar range: cmin={cmin}, cmax={cmax}, ci={ci}")
print(f"Stippling mode: {STIPPLE_MODE}")
if STIPPLE_MODE == "probability":
    print(f"Probability threshold: {PROB_STIPPLE_THRESHOLD}")

# ---------------------------------------------------------------------
# Helper for adding stippling
# ---------------------------------------------------------------------
def add_bootstrap_stippling(
    ax,
    ds_boot,
    lead,
    case_key,
    init_month,
):
    """
    Add stippling to ax based on bootstrap output.
    """
    if lead not in ds_boot.L.values:
        print(f"{case_key}, init={init_month}, L={lead}: lead not in bootstrap file")
        return

    if STIPPLE_MODE == "significant":
        if "left_better" not in ds_boot:
            print(
                f"{case_key}, init={init_month}, L={lead}: "
                "no left_better variable"
            )
            return

        sig = ds_boot["left_better"].sel(L=lead).astype(bool)

    elif STIPPLE_MODE == "probability":
        if "prob_left_lower_rmse" not in ds_boot:
            print(
                f"{case_key}, init={init_month}, L={lead}: "
                "no prob_left_lower_rmse variable"
            )
            return

        sig = (
            ds_boot["prob_left_lower_rmse"].sel(L=lead)
            >= PROB_STIPPLE_THRESHOLD
        )

    else:
        raise ValueError(f"Unknown STIPPLE_MODE: {STIPPLE_MODE}")

    sig_sub = sig.isel(
        lat=slice(None, None, STIPPLE_STRIDE_LAT),
        lon=slice(None, None, STIPPLE_STRIDE_LON),
    )

    lon2d, lat2d = np.meshgrid(sig_sub.lon.values, sig_sub.lat.values)
    mask = sig_sub.values.astype(bool)

    n_points = int(mask.sum())
    print(
        f"{case_key}, init={init_month}, L={lead}: "
        f"stipple points after subsampling = {n_points}"
    )

    if n_points == 0:
        return

    ax.scatter(
        lon2d[mask],
        lat2d[mask],
        s=STIPPLE_MARKER_SIZE,
        c="k",
        marker=".",
        alpha=STIPPLE_ALPHA,
        linewidths=0,
        transform=ccrs.PlateCarree(),
        zorder=20,
    )

# ---------------------------------------------------------------------
# Combine May and November into one figure, following the 1_ref* layout.
# Columns are grouped by initialization month; each month contains both
# E3SM-minus-CESM comparisons.
# ---------------------------------------------------------------------
plot_columns = [
    (init_month, case_key)
    for init_month in plot_months
    for case_key in diff_cases
]
nrows = len(plot_leads)
ncols = len(plot_columns)
fig = plt.figure(
    figsize=(compare_col_width * ncols, compare_row_height * nrows)
)
cntr = None
top_row_axes = {}

for i, lead in enumerate(plot_leads):
    for j, (init_month, case_key) in enumerate(plot_columns):
        ds_diff = direct_rmse_diff_by_case_month[case_key][init_month]
        rmse_diff = ds_diff["rmse_diff"].sel(L=lead)
        title = lead_label(init_month, lead)
        subplot = i * ncols + j + 1

        ax, cntr = maps.map_pcolor_global_subplot(
            fig,
            rmse_diff,
            rmse_diff.lon,
            rmse_diff.lat,
            ci, cmin, cmax,
            title,
            nrows, ncols, subplot,
            proj, cmap="RdBu_r", cutoff=0.5,
            fontsize=compare_panel_title_size,
        )
        if i == 0:
            top_row_axes[j] = ax

        if (
            PLOT_BOOTSTRAP_STIPPLING
            and init_month in bootstrap_rmse_diff_by_case_month.get(case_key, {})
        ):
            add_bootstrap_stippling(
                ax,
                bootstrap_rmse_diff_by_case_month[case_key][init_month],
                lead,
                case_key,
                init_month,
            )

        ax.set_xticks(compare_lon_ticks, crs=ccrs.PlateCarree())
        ax.set_yticks(compare_lat_ticks, crs=ccrs.PlateCarree())
        ax.set_xticklabels(compare_lon_tick_labels)
        ax.set_yticklabels(compare_lat_tick_labels)
        ax.tick_params(
            labelsize=compare_lat_lon_label_size,
            length=2.5,
            width=0.5,
            top=False,
            right=False,
            labelbottom=(i == nrows - 1),
            labelleft=(j == 0),
        )
        ax.gridlines(
            crs=ccrs.PlateCarree(), linewidth=0.35,
            color="0.35", alpha=0.35, linestyle="-", draw_labels=False,
        )

fig.suptitle(
    f"{cfg['plot_name']} direct-RMSE difference: E3SM minus CESM-SMYLE",
    fontsize=compare_suptitle_size,
    fontweight=compare_fweight,
    y=compare_suptitle_y,
)
fig.tight_layout(rect=compare_tight_layout_rect, h_pad=0.75, w_pad=0.45)
fig.subplots_adjust(hspace=compare_hspace, wspace=compare_wspace)

# Model-comparison headers sit above the lead/season panel titles.
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
title_tops = [
    ax.title.get_window_extent(renderer=renderer)
    .transformed(fig.transFigure.inverted()).y1
    for ax in top_row_axes.values()
]
column_header_y = min(
    compare_column_header_y_max,
    max(title_tops) + compare_column_header_pad,
)
for j, (_, case_key) in enumerate(plot_columns):
    ax_bbox = top_row_axes[j].get_position()
    case_label = model_display_name.get(case_key, case_key)
    fig.text(
        0.5 * (ax_bbox.x0 + ax_bbox.x1),
        column_header_y,
        f"{case_label} - CESM-SMYLE",
        ha="center",
        va="bottom",
        fontsize=compare_column_header_size,
        fontweight=compare_fweight,
    )

cbar_ax = fig.add_axes(compare_colorbar_axes)
cbar = fig.colorbar(cntr, cax=cbar_ax, orientation="horizontal")
cbar.set_ticks(diff_ticks)
cbar.ax.xaxis.set_major_formatter(
    mticker.FormatStrFormatter(f"%.{diff_decimals}f")
)
cbar.set_label(
    f"RMSE difference ({field}, {metric_units})",
    fontsize=compare_colorbar_label_size,
    fontweight=compare_fweight,
)
cbar.ax.tick_params(labelsize=compare_colorbar_tick_size)

stipple_tag = (
    f"prob{int(PROB_STIPPLE_THRESHOLD * 100):02d}_markers"
    if PLOT_BOOTSTRAP_STIPPLING else "no_markers"
)
figname = figure_filename(field, "rmse_diff_global", ext=figfmt)
figpath = FIGURE_OUTDIR / figname
mov.save_figure(
    fig, figpath,
    mode="",
    metric="rmse_compare",
    title=f"RMSE Comparison: {field}",
    caption="RMSE comparison across experiments",
    dpi=dpi,
)
print("Saved figure:", figpath)
plt.show()


### Requested four-column direct-RMSE difference comparison

This plots four requested direct-RMSE difference columns for each initialization month:

- `E3SM-FOSIRL - CESM-SMYLE`, labeled as `E3SMv3-FOSIRL - CESM-SMYLE`
- `E3SM-Reanalysis - CESM-SMYLE`, labeled as `E3SMv3-Reanalysis - CESM-SMYLE`
- `E3SM-4DEnVarOcn - CESM-SMYLE`, labeled as `E3SMv3-4DEnVarOcn - CESM-SMYLE`
- `E3SM-Reanalysis - E3SM-FOSIRL`, labeled as `E3SMv3-Reanalysis - E3SMv3-FOSIRL`


In [ ]:
%%time
# Plot requested four-column direct RMSE differences.
#
# Columns:
#   1. E3SM-FOSIRL minus CESM-SMYLE, labeled as E3SMv3-FOSIRL - CESM-SMYLE
#   2. E3SM-Reanalysis minus CESM-SMYLE, labeled as E3SMv3-Reanalysis - CESM-SMYLE
#   3. E3SM-4DEnVarOcn minus CESM-SMYLE, labeled as E3SMv3-4DEnVarOcn - CESM-SMYLE
#   4. E3SM-Reanalysis minus E3SM-FOSIRL, labeled as E3SMv3-Reanalysis - E3SMv3-FOSIRL
#
# Negative values mean the left-hand model in the column title has smaller
# model-vs-observation direct RMSE than the right-hand model.

import cartopy.crs as ccrs
import os
import numpy as np
import xarray as xr

# ---------------------------------------------------------------------
# User-controlled setup parameters
# ---------------------------------------------------------------------
comparison_specs = [
    {
        "key": "E3SM-FOSIRL_minus_CESM-SMYLE",
        "left_model": "E3SM-FOSIRL",
        "right_model": "CESM-SMYLE",
        "left_label": "E3SMv3-FOSIRL",
        "right_label": "CESM-SMYLE",
    },
    {
        "key": "E3SM-Reanalysis_minus_CESM-SMYLE",
        "left_model": "E3SM-Reanalysis",
        "right_model": "CESM-SMYLE",
        "left_label": "E3SMv3-Reanalysis",
        "right_label": "CESM-SMYLE",
    },
    {
        "key": "E3SM-4DEnVarOcn_minus_CESM-SMYLE",
        "left_model": "E3SM-4DEnVarOcn",
        "right_model": "CESM-SMYLE",
        "left_label": "E3SMv3-4DEnVarOcn",
        "right_label": "CESM-SMYLE",
    },
    {
        "key": "E3SM-Reanalysis_minus_E3SM-FOSIRL",
        "left_model": "E3SM-Reanalysis",
        "right_model": "E3SM-FOSIRL",
        "left_label": "E3SMv3-Reanalysis",
        "right_label": "E3SMv3-FOSIRL",
    },
]

# Colorbar range for RMSE difference:
#   negative = left-hand model lower RMSE than right-hand model
#   positive = left-hand model higher RMSE than right-hand model
# Use the RMSE-difference colorbar selected with `field` above.
cmin = float(diff_levels[0])
cmax = float(diff_levels[-1])
ci = float(np.diff(diff_levels)[0])

# Typography. Change only fontz to scale the whole figure proportionally.
fontz = 16
compare_fweight = "bold"
compare_suptitle_size = 1.10 * fontz
compare_column_header_size = 1.00 * fontz
compare_panel_title_size = 1.00 * fontz
compare_lead_label_size = 0.90 * fontz
compare_lat_lon_label_size = 0.95 * fontz
compare_colorbar_label_size = 1.00 * fontz
compare_colorbar_tick_size = 0.95 * fontz
compare_suptitle_y = 0.988
compare_column_header_pad = 0.006
compare_column_header_y_max = 0.975

# Figure layout.
compare_fig_width_per_col = 6.0
compare_fig_height_per_row = 2.8
compare_tight_layout_rect = [0, 0.06, 1, 0.965]
compare_subplot_bottom = 0.078
compare_subplot_wspace = 0.05
compare_colorbar_axes = [0.14, 0.04, 0.72, 0.010]  # [left, bottom, width, height]
compare_colorbar_orientation = "horizontal"
compare_colorbar_outline_width = 0.6
compare_colorbar_tick_length = 2.0
compare_colorbar_tick_width = 0.6

# Map shading and grid styling.
compare_projection = ccrs.PlateCarree()
compare_cmap = "RdBu_r"
compare_cutoff = 0.5
compare_gridline_width = 0.025 * fontz
compare_gridline_color = "0.35"
compare_gridline_alpha = 0.35
compare_gridline_style = "-"

# Finite-ensemble resampling markers following 1_ref*. They are available
# only when the compared ensembles have unequal member counts.
PLOT_MATCHED_ENSEMBLE_MARKERS = True
COMPARE_PROBABILITY_THRESHOLD = 0.9
compare_marker_stride = 5
compare_open_marker_size = 12
compare_filled_marker_size = 5
compare_marker_linewidth = 0.65
compare_marker_color = "black"
BOOT_RMSE_OUTDIR = globals().get(
    "BOOT_RMSE_OUTDIR",
    os.path.join(DIRECT_RMSE_OUTDIR, "bootstrap_rmse_diff"),
)
NBOOT = globals().get("NBOOT", 1000)

# Publication-style sparse map tick labels.
compare_lon_ticks = [-160, -80, 0, 80, 160]
compare_lon_tick_labels = [
    r"160$^\circ$W",
    r"80$^\circ$W",
    r"0$^\circ$",
    r"80$^\circ$E",
    r"160$^\circ$E",
]
compare_lat_ticks = [-60, -30, 0, 30, 60]
compare_lat_tick_labels = [
    r"60$^\circ$S",
    r"30$^\circ$S",
    r"0$^\circ$",
    r"30$^\circ$N",
    r"60$^\circ$N",
]
compare_tick_length = 0.25 * fontz
compare_tick_width = 0.06 * fontz

# ---------------------------------------------------------------------
# Load direct RMSE files if needed
# ---------------------------------------------------------------------
def direct_rmse_filename(model, init_month):
    year_tag = "-".join(str(y) for y in SMYLE_MATCHED_YEARS_BY_MONTH[init_month])
    return os.path.join(
        DIRECT_RMSE_OUTDIR,
        f"{safe_model_name(model)}_{field}_direct_rmse_init{init_month:02d}_years_{year_tag}.nc",
    )


required_models = sorted(
    {spec["left_model"] for spec in comparison_specs}
    | {spec["right_model"] for spec in comparison_specs}
)

if "direct_rmse_by_requested_model_month" not in globals():
    direct_rmse_by_requested_model_month = {model: {} for model in required_models}
else:
    for model in required_models:
        direct_rmse_by_requested_model_month.setdefault(model, {})

for model in required_models:
    for init_month in init_months:
        if init_month in direct_rmse_by_requested_model_month[model]:
            continue

        # Prefer already-computed notebook dictionaries when available.
        ds_rmse = None
        if model == "CESM-SMYLE" and "direct_rmse_smyle_by_month" in globals():
            ds_rmse = direct_rmse_smyle_by_month.get(init_month)
        elif "direct_rmse_e3sm_by_case_month" in globals():
            ds_rmse = direct_rmse_e3sm_by_case_month.get(model, {}).get(init_month)

        # Fall back to the saved direct-RMSE NetCDF files.
        if ds_rmse is None:
            fpath = direct_rmse_filename(model, init_month)

            if not os.path.exists(fpath):
                print("Direct RMSE file not found:", fpath)
                continue

            ds_rmse = normalize_direct_rmse_leads(xr.open_dataset(fpath).load())
            print("Loaded direct RMSE file:", fpath)

        direct_rmse_by_requested_model_month[model][init_month] = (
            normalize_direct_rmse_leads(ds_rmse)
        )

# ---------------------------------------------------------------------
# Build requested RMSE differences
# ---------------------------------------------------------------------
requested_rmse_diff_by_column_month = {
    spec["key"]: {}
    for spec in comparison_specs
}

for spec in comparison_specs:
    for init_month in init_months:
        left_available = init_month in direct_rmse_by_requested_model_month[spec["left_model"]]
        right_available = init_month in direct_rmse_by_requested_model_month[spec["right_model"]]

        if not (left_available and right_available):
            continue

        requested_rmse_diff_by_column_month[spec["key"]][init_month] = (
            rmse_compare_helper.direct_rmse_difference(
                direct_rmse_by_requested_model_month[spec["left_model"]][init_month],
                direct_rmse_by_requested_model_month[spec["right_model"]][init_month],
            )
        )

# Load finite-ensemble probabilities produced by the preceding comparison
# cell. Missing files disable markers for equal-size ensemble comparisons.
matched_bootstrap_by_column_month = {
    spec["key"]: {} for spec in comparison_specs
}

for spec in comparison_specs:
    safe_left = safe_model_name(spec["left_model"])
    safe_right = safe_model_name(spec["right_model"])
    for init_month in init_months:
        year_tag = "-".join(str(y) for y in SMYLE_MATCHED_YEARS_BY_MONTH[init_month])
        boot_path = os.path.join(
            BOOT_RMSE_OUTDIR,
            f"{safe_left}_minus_{safe_right}_{field}_finite_ensemble_resampling_"
            f"init{init_month:02d}_years_{year_tag}_niter{NBOOT}.nc",
        )
        if os.path.exists(boot_path):
            matched_bootstrap_by_column_month[spec["key"]][init_month] = (
                normalize_direct_rmse_leads(xr.open_dataset(boot_path).load())
            )
        elif PLOT_MATCHED_ENSEMBLE_MARKERS:
            print("Matched-ensemble bootstrap file not found:", boot_path)

# ---------------------------------------------------------------------
# Determine months/leads to plot
# ---------------------------------------------------------------------
plot_months = [
    init_month for init_month in init_months
    if all(
        init_month in requested_rmse_diff_by_column_month[spec["key"]]
        for spec in comparison_specs
    )
]

plot_leads = [
    lead for lead in map_plot_leads
    if all(
        lead in requested_rmse_diff_by_column_month[spec["key"]][init_month].L.values
        for spec in comparison_specs
        for init_month in plot_months
    )
]

if len(plot_months) == 0:
    raise ValueError("No initialization months available for plotting.")

if len(plot_leads) == 0:
    raise ValueError("No leads available for plotting.")

print("plot_months:", plot_months)
print("plot_leads:", plot_leads)

proj = compare_projection

# ---------------------------------------------------------------------
# Colorbar range
# ---------------------------------------------------------------------
print(f"Colorbar range: cmin={cmin}, cmax={cmax}, ci={ci}")

# ---------------------------------------------------------------------
# Combine all initialization months into one figure
# ---------------------------------------------------------------------
plot_columns = [
    (init_month, spec)
    for init_month in plot_months
    for spec in comparison_specs
]

nrows = len(plot_leads)
ncols = len(plot_columns)

fig = plt.figure(
    figsize=(compare_fig_width_per_col * ncols, compare_fig_height_per_row * nrows)
)
cntr = None
top_row_axes = {}

for i, lead in enumerate(plot_leads):
    for j, (init_month, spec) in enumerate(plot_columns):
        ds_diff = requested_rmse_diff_by_column_month[spec["key"]][init_month]

        leadstr = lead_label(init_month, lead)
        title = leadstr
        subplot = i * ncols + j + 1
        rmse_diff = ds_diff["rmse_diff"].sel(L=lead)

        ax, cntr = maps.map_pcolor_global_subplot(
            fig,
            rmse_diff,
            rmse_diff.lon,
            rmse_diff.lat,
            ci,
            cmin,
            cmax,
            title,
            nrows,
            ncols,
            subplot,
            proj,
            cmap=compare_cmap,
            cutoff=compare_cutoff,
            fontsize=compare_panel_title_size,
        )
        if i == 0:
            top_row_axes[j] = ax

        ax.set_xticks(compare_lon_ticks, crs=ccrs.PlateCarree())
        ax.set_yticks(compare_lat_ticks, crs=ccrs.PlateCarree())
        ax.set_xticklabels(compare_lon_tick_labels)
        ax.set_yticklabels(compare_lat_tick_labels)

        ax.tick_params(
            labelsize=compare_lat_lon_label_size,
            length=compare_tick_length,
            width=compare_tick_width,
            top=False,
            right=False,
            labelbottom=(i == nrows - 1),
            labelleft=(j == 0),
        )

        ax.gridlines(
            crs=ccrs.PlateCarree(),
            linewidth=compare_gridline_width,
            color=compare_gridline_color,
            alpha=compare_gridline_alpha,
            linestyle=compare_gridline_style,
            draw_labels=False,
        )

        boot_ds = matched_bootstrap_by_column_month[spec["key"]].get(init_month)
        if (
            PLOT_MATCHED_ENSEMBLE_MARKERS
            and boot_ds is not None
            and lead in boot_ds.L.values
        ):
            prob_left_better = boot_ds["prob_left_lower_rmse"].sel(L=lead)
            left_better = prob_left_better >= COMPARE_PROBABILITY_THRESHOLD
            right_better = prob_left_better <= (1 - COMPARE_PROBABILITY_THRESHOLD)

            lon2d, lat2d = np.meshgrid(rmse_diff.lon, rmse_diff.lat)
            display_lon = lon2d[::compare_marker_stride, ::compare_marker_stride]
            display_lat = lat2d[::compare_marker_stride, ::compare_marker_stride]

            right_display = np.asarray(right_better)[
                ::compare_marker_stride, ::compare_marker_stride
            ]
            left_display = np.asarray(left_better)[
                ::compare_marker_stride, ::compare_marker_stride
            ]

            # Open circles: right-hand model robustly has lower RMSE.
            ax.scatter(
                display_lon[right_display], display_lat[right_display],
                facecolor="none", edgecolor=compare_marker_color,
                s=compare_open_marker_size,
                linewidth=compare_marker_linewidth,
                transform=ccrs.PlateCarree(), zorder=20,
            )
            # Filled dots: left-hand model robustly has lower RMSE.
            ax.scatter(
                display_lon[left_display], display_lat[left_display],
                facecolor=compare_marker_color, edgecolor=compare_marker_color,
                s=compare_filled_marker_size, linewidth=0,
                transform=ccrs.PlateCarree(), zorder=20,
            )

            right_fraction = area_weighted_mask_fraction(right_better)
            left_fraction = area_weighted_mask_fraction(left_better)
            ax.text(
                0.02, 0.04,
                f"open/right {right_fraction * 100:.1f}% | "
                f"filled/left {left_fraction * 100:.1f}%",
                transform=ax.transAxes,
                ha="left", va="bottom",
                fontsize=compare_lead_label_size * 0.75,
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="lightgray", alpha=0.85, linewidth=0.8),
                zorder=21,
            )

fig.suptitle(
    f"{cfg['plot_name']} direct-RMSE difference comparison",
    fontsize=compare_suptitle_size,
    fontweight=compare_fweight,
    y=compare_suptitle_y,
)
fig.tight_layout(rect=compare_tight_layout_rect)
fig.subplots_adjust(bottom=compare_subplot_bottom, wspace=compare_subplot_wspace)

# Model-comparison headers sit above the lead/season panel titles.
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
title_tops = [
    ax.title.get_window_extent(renderer=renderer)
    .transformed(fig.transFigure.inverted()).y1
    for ax in top_row_axes.values()
]
column_header_y = min(
    compare_column_header_y_max,
    max(title_tops) + compare_column_header_pad,
)
for j, (_, spec) in enumerate(plot_columns):
    ax_bbox = top_row_axes[j].get_position()
    fig.text(
        0.5 * (ax_bbox.x0 + ax_bbox.x1),
        column_header_y,
        f"{spec['left_label']} - {spec['right_label']}",
        ha="center",
        va="bottom",
        fontsize=compare_column_header_size,
        fontweight=compare_fweight,
    )

cbar_ax = fig.add_axes(compare_colorbar_axes)
cbar = fig.colorbar(cntr, cax=cbar_ax, orientation=compare_colorbar_orientation)
cbar.set_ticks(diff_ticks)
cbar.ax.xaxis.set_major_formatter(
    mticker.FormatStrFormatter(f"%.{diff_decimals}f")
)

cbar.set_label(
    f"RMSE difference ({field}, {metric_units})",
    fontsize=compare_colorbar_label_size,
    fontweight=compare_fweight,
)

cbar.outline.set_linewidth(compare_colorbar_outline_width)
cbar.ax.tick_params(
    labelsize=compare_colorbar_tick_size,
    length=compare_colorbar_tick_length,
    width=compare_colorbar_tick_width,
)
cbar.solids.set_edgecolor("face")

month_tag = "_".join(f"init{init_month:02d}" for init_month in plot_months)
year_tag = "_".join(
    f"init{init_month:02d}_years_"
    + "-".join(str(y) for y in SMYLE_MATCHED_YEARS_BY_MONTH[init_month])
    for init_month in plot_months
)

figname = figure_filename(field, "rmse_diff_compare", ext=figfmt)

figpath = FIGURE_OUTDIR / figname
mov.save_figure(
    fig, figpath,
    mode="",
    metric="rmse_compare",
    title=f"RMSE Comparison: {field}",
    caption="RMSE comparison across experiments",
    dpi=dpi,
)
plt.show()
print("Saved combined figure:", figpath)
